### Query NewsAPI for multiple keywords and append results (topics | count | prevalence) from NewsAPI's indexed corpus

As an example, we will try these topics (with significant keywords) :

[
    "energy",
    "carbon",
    "nuclear",
    "green"
]

In [2]:
import requests
import pandas as pd

API_KEY = "6ebfd5fbe709457c8aca7a519699ed98"

queries = {
    "energy":
        '("electric vehicle" OR EV)',

    "carbon":
        '("gasoline vehicle" OR petrol OR "carbone emission")',

    "nuclear":
        '("nuclear power" OR "nuclear energy")',

    "green":
        '("renewable energy" OR "clean energy" OR wind OR solar)',
} # these can be changed for significant keywords and subjects depending the researcher's goals

results = []

for topic, query in queries.items():

    r = requests.get(
        "https://newsapi.org/v2/everything",
        params={
            "q": query,
            "language": "en",
            "from" : "2026-06-15", # kept under a month back because of the free-tier date restriction
            "to"   : "2026-06-25", # change to suit the researcher's needs
            "sortBy": "publishedAt",
            "pageSize": 1,
            "apiKey": API_KEY
        }
    ).json()

    results.append({
        "topic": topic,
        "count": r["totalResults"]
    })

df = pd.DataFrame(results)
df["share"] = df["count"] / df["count"].sum()
print(df)

     topic  count     share
0   energy   1256  0.181660
1   carbon    486  0.070292
2  nuclear    434  0.062771
3    green   4738  0.685276


### If needed, we can also query for examples of recent articles containing some of our keywords and topics.
We can check the type of articles that come out recently, qualitatively compared to those we have in our database.

In [3]:
import requests
import pandas as pd
from datetime import datetime, timedelta

API_KEY = "6ebfd5fbe709457c8aca7a519699ed98"

base_url = "https://newsapi.org/v2/everything"

queries = {
    "green":
        '("renewable energy" OR "clean energy" OR wind OR solar)'
}

all_articles = []

for topic, query in queries.items():

    params = {
        "q": query,
        "language": "en",
        "sortBy": "publishedAt",
        "from": (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d"), # kept under a month back because of the free-tier date restriction
        "pageSize": 100,
        "apiKey": API_KEY
    }

    response = requests.get(base_url, params=params)
    data = response.json()

    for article in data.get("articles", []):
        all_articles.append({
            "topic": topic,
            "title": article["title"],
            "description": article["description"],
            "source": article["source"]["name"],
            "publishedAt": article["publishedAt"],
            "url": article["url"]
        })

df = pd.DataFrame(all_articles)

print(df.shape)
df.head()

(91, 6)


,topic,title,description,source,publishedAt,url
0,green,VOC Port emerges as model for sustainable mari...,Leading the charge in India's sustainable mari...,The Times of India,2026-06-24T09:22:01Z,https://economictimes.indiatimes.com/industry/...
1,green,"Refurb EcoFlow DELTA Pro 3,600Wh Power Station...","Apply coupon code ""SUNSHINE20"" at checkout to ...",Dealnews.com,2026-06-24T09:21:10Z,https://www.dealnews.com/Refurb-Eco-Flow-DELTA...
2,green,Your Home Could Help Solve AI’s Growing Power ...,"Tesla, Sunrun and Renew Home plan to tap solar...",Biztoc.com,2026-06-24T09:21:06Z,https://biztoc.com/x/118438f090066238
3,green,The parallels between Morocco and Israel are i...,With the Western Sahara stalemate and Morocco-...,Israelnationalnews.com,2026-06-24T09:21:00Z,https://www.israelnationalnews.com/news/429125
4,green,Phillies' Cristopher Sanchez is a long shot to...,Philadelphia Phillies left-hander Cristopher S...,Sporting News,2026-06-24T09:19:59Z,https://www.sportingnews.com/us/mlb/philadelph...


# Conclusion

In this notebook, we:

- Tried to query NewsAPI for keyword frequency in recent news articles publications
- Tried to query for examples of such articles

We are now ready to compare to keywords and specific topics in our datasets. If they are very prevalent in recent publications and mostly absent of our datasets, we can conclude that the models trained on our datasets will have performance issues generalizing to the new subjects in recent unseen data.